Aula4- Teste ROC Curve e Kruskall Wallis

In [60]:
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt

# Ler o ficheiro: separador ';', decimal ',' e saltar a linha vazia (linha 2)
df = pd.read_csv(
    r"C:/Users/josem/OneDrive/Ambiente de Trabalho/Zé Miguel/UNIVERSIDADE/5ANO/Rodrigo_Jose/AC/datasets/CORK_STOPPERS.csv",
    sep=";",
    decimal=",",
    skiprows=[1]
)

# Definir as características a analisar
features = ["ART", "N.1", "PRT", "ARM", "PRM",
            "ARTG", "NG", "PRTG", "RAAR", "RAN"]

# Definir as categorias de qualidade
quality = ["Super", "Average", "Poor"]

# Converter os códigos numéricos da coluna C em categorias
df["C"] = df["C"].map({
    1: quality[0],
    2: quality[1],
    3: quality[2]
})

# Criar um violin plot para cada característica
for i in features:
    fig = px.violin(
        data_frame=df,
        x="C",
        y=i,
        color="C",
        box=True,
        points="all",
        title=f"Distribuição de {i} por qualidade"
    )

    fig.show()

Violin plot per feature

Teste de Kruskall Wallis

In [61]:
# Número de amostras
N = df.shape[0]

# Rank médio global
R_medio_global = (N+1) / 2

resultados = []

for feat in features:
    # Ranks da feature
    ranks = df[feat].rank()

    soma = 0
    for j in quality:
        # Linhas da classe j
        linhas = df["C"] == j
        ni = linhas.sum()

        # Média dos ranks da classe j
        Ri = ranks[linhas].mean()

        soma = soma + ni * (Ri - R_medio_global) ** 2

    # Fórmula de H
    H = 12 / (N * (N + 1)) * soma
    resultados.append([feat, H])

# Ordenar do maior para o menor H
resultados.sort(key=lambda x: x[1], reverse=True)

for feat, H in resultados:
    print(feat, round(H, 2))

ART 121.59
PRM 117.63
PRT 115.74
ARTG 114.99
ARM 113.53
PRTG 113.1
RAAR 105.01
NG 102.77
RAN 94.12
N.1 74.47


Correlation matrix for first 5 features

In [62]:
features5 = features[:5]
corr5 = df[features5].corr()
print(corr5.round(3))

       ART    N.1    PRT    ARM    PRM
ART  1.000  0.805  0.981  0.869  0.882
N.1  0.805  1.000  0.892  0.453  0.494
PRT  0.981  0.892  1.000  0.779  0.814
ARM  0.869  0.453  0.779  1.000  0.985
PRM  0.882  0.494  0.814  0.985  1.000


ROC-AUC Curve

In [63]:
from sklearn.metrics import roc_auc_score

# Ficar só com as duas primeiras classes: Super e Average
df2 = df[df["C"].isin(["Super", "Average"])]

# Variável alvo: 1 para Average, 0 para Super
y = (df2["C"] == "Average").astype(int)

resultados = []
for feat in features:
    # AUC da feature usada como score para separar as duas classes
    auc = roc_auc_score(y, df2[feat])
    resultados.append((feat, auc))

# Ordenar da feature com maior AUC para a de menor AUC
resultados.sort(key=lambda x: x[1], reverse=True)
for feat, auc in resultados:
    print(feat, round(auc, 3))

ART 0.947
PRT 0.929
PRM 0.901
ARM 0.887
ARTG 0.867
PRTG 0.861
N.1 0.848
NG 0.839
RAAR 0.797
RAN 0.788
